# SNISA + Unlearning by Confusion (UnconLa) Example

This notebook demonstrates the combined approach of **SNISA (Sharded, Non-Isolated, Sliced, Aggregated)** training with the **UnconLa (Unlearning by Confusion)** technique using the modularized codebase.

**Goal:** To train an ensemble model where each sub-model is trained on a separate shard of the data (SNISA), and during the training of each sub-model, noise is periodically added to its weights (UnconLa). This aims to make the model less sensitive to specific training samples, potentially improving unlearning capabilities.

**Steps:**
1.  **Setup:** Initialize TensorFlow, configure GPU/CPU usage, and import necessary modules.
2.  **Configuration:** Define hyperparameters like batch size, number of shards, forget fraction, confusion noise level, and training epochs.
3.  **Load Data:** Load CIFAR-10, normalize it, and split it into training, validation, test, forget, and retain sets. Further partition the retain and forget sets into the specified number of shards.
4.  **Load Base Model:** Load a pre-trained ResNet18 model that serves as the starting point.
5.  **Create Confused Base:** Clone the base model and apply initial weight confusion (noise). This confused model will be the starting point for training each shard model.
6.  **SNISA + UnconLa Training:** Loop through each shard:
    *   Clone the confused base model.
    *   Train the model on the shard's retain data for a few epochs.
    *   Apply confusion (add noise).
    *   Repeat training and confusion steps.
    *   Store the final trained model for the shard.
7.  **Build Ensemble:** Create an ensemble model that averages the predictions of all trained shard models.
8.  **Evaluate Accuracy:** Assess the ensemble's performance on the retain, forget, and test datasets.
9.  **Perform MIA:** Conduct a Membership Inference Attack to gauge how easily one can distinguish between samples the model was trained on (forget set) versus unseen samples (test set). Lower MIA accuracy suggests better unlearning.
10. **Plot Results:** Visualize the loss distribution for forget vs. test sets and optionally plot the training history for individual shards.
11. **Save Results:** Store the key metrics (accuracies, MIA score) for analysis.

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers # Added for ensemble building
import matplotlib.pyplot as plt
import pickle
from tqdm import tqdm
import sys

# Add project root to path to allow module imports
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Import project modules
from utils.setup import set_device
from data.cifar10 import load_cifar10, normalize, get_dataset_partitions, AUTOTUNE
from models.resnet18 import build_resnet18_model
from unlearning.layer_reset import vision_confuser
from evaluation.metrics import evaluate_model, compute_losses
from evaluation.mia import perform_mia_on_forget_test
from utils.helpers import clone_model, change_model_names
from utils.plotting import plot_loss_distribution, plot_mia_comparison, plot_training_history # Added plot_training_history


2025-05-01 15:28:58.208981: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-01 15:28:58.209165: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-01 15:28:58.278270: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-01 15:28:58.421148: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-01 15:29:00.241417: W tensorflow/compiler/tf2

Using TensorFlow backend


/home/roman/anaconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Setup Device and Configuration

In [ ]:
# Configure TensorFlow device settings (GPU/CPU, mixed precision, JIT)
strategy, physical_devices = set_device(mixed_precision=True, set_jit=True)
print(f'TensorFlow Version: {tf.__version__}')
print(f'Using strategy: {strategy}')

# --- Configuration Hyperparameters ---
BATCH_SIZE = 128
NUM_SHARDS = 4          # Number of shards (S in SNISA)
FORGET_FRACTION = 0.1   # Fraction of training data designated as 'forget' set
CONFUSION_STD = 0.08    # Standard deviation for the Gaussian noise added during confusion
# Epochs for each training stage within a shard's training cycle:
# [train_epochs_1, train_epochs_2, train_epochs_3]
# The cycle is: train -> confuse -> train -> confuse -> train
EPOCHS_PER_STAGE = [1, 1, 1, 1, 2, 4] # Total epochs per shard
TOTAL_EPOCHS_PER_SHARD = sum(EPOCHS_PER_STAGE)
SEED = 42               # Random seed for reproducibility
BASE_MODEL_PATH = '../resnet18_cifar10.keras' # Path to the pre-trained base model (relative to notebook)
RESULTS_DIR = './results' # Directory to save results and models
MODELS_DIR = os.path.join(RESULTS_DIR, f'snisa_unconla_S{NUM_SHARDS}_std{CONFUSION_STD}_models') # Dir for shard models

# Create directories if they don't exist
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print(f'Results will be saved in: {RESULTS_DIR}')
print(f'Shard models will be saved in: {MODELS_DIR}')


Detected devices:
- /device:CPU:0 (Memory Limit: 268435456 bytes)
- /device:GPU:0 (Memory Limit: 4254072832 bytes)
Using device type: GPU
JIT Compilation: Enabled
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 2060, compute capability 7.5
Mixed Precision: Enabled (mixed_float16)
Memory growth enabled for /physical_device:GPU:0
Using default TensorFlow strategy.
TensorFlow Version: 2.15.1
Using strategy: <tensorflow.python.distribute.distribute_lib._DefaultDistributionStrategy object at 0x7f186bf291e0>
Results will be saved in: ./results
Shard models will be saved in: ./results/snisa_unconla_S4_std0.8_models
Mixed Precision: Enabled (mixed_float16)
Memory growth enabled for /physical_device:GPU:0
Using default TensorFlow strategy.
TensorFlow Version: 2.15.1
Using strategy: <tensorflow.python.distribute.distribute_lib._Defaul

2025-05-01 15:29:07.016867: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-01 15:29:07.268525: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-01 15:29:07.268655: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-01 15:29:07.593190: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-01 15:29:07.593523: I external/local_xla/xla/stream_executor

## 2. Load and Prepare Data
Load CIFAR-10. Split into train/val/test. Further split train into forget/retain based on `FORGET_FRACTION`. Finally, partition the retain and forget sets into `NUM_SHARDS`.

In [3]:
# Load CIFAR-10 datasets: train, validation, test, forget, retain
# Also get unbatched, normalized versions for partitioning and MIA
(train_ds, val_ds, test_ds, forget_ds, retain_ds, 
 forget_set_normalized, retain_set_normalized, test_ds_normalized) = load_cifar10(
    batch_size=BATCH_SIZE, forget_fraction=FORGET_FRACTION, seed=SEED
)

# Partition the unbatched, normalized retain and forget sets into shards
print('\nPartitioning retain set...')
retain_shards_unbatched = get_dataset_partitions(retain_set_normalized, NUM_SHARDS)
print('\nPartitioning forget set...')
forget_shards_unbatched = get_dataset_partitions(forget_set_normalized, NUM_SHARDS)

# Create lists of *batched* datasets for training and evaluation within the loop
# These are derived from the unbatched shards
rt_datasets = [] # List of batched retain shard datasets
ft_datasets = [] # List of batched forget shard datasets
for i in range(NUM_SHARDS):
    # Shuffle, batch, and prefetch each retain shard for training efficiency
    rt_shard = retain_shards_unbatched[i].shuffle(buffer_size=8*BATCH_SIZE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    # Batch and prefetch each forget shard (shuffling not needed for evaluation)
    ft_shard = forget_shards_unbatched[i].batch(BATCH_SIZE).prefetch(AUTOTUNE)
    rt_datasets.append(rt_shard)
    ft_datasets.append(ft_shard)

print(f'\nCreated {len(rt_datasets)} batched retain shards and {len(ft_datasets)} batched forget shards.')


Loading CIFAR-10 dataset...
Original training data shape: (50000, 32, 32, 3)
Original held-out data shape: (10000, 32, 32, 3)
Original training data shape: (50000, 32, 32, 3)
Original held-out data shape: (10000, 32, 32, 3)


2025-05-01 15:29:08.460537: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-01 15:29:08.460670: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-01 15:29:08.460729: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-01 15:29:08.461206: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-01 15:29:08.461222: I tensorflow/core/common_runtime/gpu/gpu

Validation set size: 8000
Test set size: 2000
Full training dataset size: 50000
Full training dataset size: 50000
Forget set size: 5000
Retain set size: 45000
CIFAR-10 datasets prepared.

Partitioning retain set...
Partitioning dataset of size 45000 into 4 shards...
  Shard 0 size: 11250
  Shard 1 size: 11250
  Shard 2 size: 11250
  Shard 3 size: 11250

Partitioning forget set...
Partitioning dataset of size 5000 into 4 shards...
  Shard 0 size: 1250
  Shard 1 size: 1250
  Shard 2 size: 1250
  Shard 3 size: 1250

Created 4 batched retain shards and 4 batched forget shards.
Forget set size: 5000
Retain set size: 45000
CIFAR-10 datasets prepared.

Partitioning retain set...
Partitioning dataset of size 45000 into 4 shards...
  Shard 0 size: 11250
  Shard 1 size: 11250
  Shard 2 size: 11250
  Shard 3 size: 11250

Partitioning forget set...
Partitioning dataset of size 5000 into 4 shards...
  Shard 0 size: 1250
  Shard 1 size: 1250
  Shard 2 size: 1250
  Shard 3 size: 1250

Created 4 batch

## 3. Load Pre-trained Model
Load the initial ResNet18 model trained on the full dataset.

In [4]:
# Check if the base model file exists
if not os.path.exists(BASE_MODEL_PATH):
# print(f'Error: Base model file not found at {BASE_MODEL_PATH}')
# print('Please ensure the pre-trained model exists or train one first.')
# print('You might need to run the Base_model.ipynb notebook first.')
    # Optionally, add code here to train a base model if it doesn't exist
    print('Building and training a new base model...')
    with strategy.scope():
        original_model = build_resnet18_model(strategy)
    # Define epochs and train
    history = original_model.fit(train_ds, validation_data=val_ds, epochs=20, verbose=1)
    original_model.save(BASE_MODEL_PATH)
    print(f'New base model trained and saved to {BASE_MODEL_PATH}')
    # raise FileNotFoundError(f"Base model not found: {BASE_MODEL_PATH}") # Stop execution if model is missing
else:
    print(f'Loading pre-trained model from {BASE_MODEL_PATH}...')
    # Load the model within the distribution strategy scope
    with strategy.scope(): 
        original_model = keras.saving.load_model(BASE_MODEL_PATH)
    print('Model loaded successfully.')
    # Display model summary
    original_model.summary()


Loading pre-trained model from ../resnet18_cifar10.keras...
Model loaded successfully.
Model: "resnet18_cifar10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 res_net_backbone (ResNetBa  (None, 1, 1, 512)         11186112  
 ckbone)                                                         
                                                                 
 global_max_pooling2d (Glob  (None, 512)               0         
 alMaxPooling2D)                                                 
                                                                 
 dense (Dense)               (None, 10)                5130      
                                                                 
Model loaded successfully.
Model: "resnet18_cifar10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 res_net_backbone (ResNetBa  (None, 1, 1, 

## 4. Create Base Confused Model (UnconLa)
Clone the original model and apply the initial layer confusion by adding Gaussian noise to the weights. This serves as the starting point for each shard's training.

In [5]:
# Clone the original model to create the base for confused training
print('Cloning the original model...')
base_confused_model = clone_model(original_model, strategy)
# Rename the cloned model and its layers to avoid conflicts
change_model_names(base_confused_model, name='resnet18_confused_base', suffix='_confused_base')

# Apply initial confusion using the vision_confuser function
print('\nApplying initial confusion to the cloned model...')
vision_confuser(base_confused_model, std=CONFUSION_STD)
print('\nInitial confusion applied.')
base_confused_model.summary()


Cloning the original model...
Cloned and re-compiled model 'resnet18_cifar10' with explicit accuracy metric.
Updated layer names for model 'res_net_backbone_confused_base' with suffix '_confused_base'.
Updated layer names for model 'resnet18_confused_base' with suffix '_confused_base'.

Applying initial confusion to the cloned model...
Applying vision confusion with std=0.8 to model 'resnet18_confused_base'...
  Found backbone layer: res_net_backbone_confused_base
Cloned and re-compiled model 'resnet18_cifar10' with explicit accuracy metric.
Updated layer names for model 'res_net_backbone_confused_base' with suffix '_confused_base'.
Updated layer names for model 'resnet18_confused_base' with suffix '_confused_base'.

Applying initial confusion to the cloned model...
Applying vision confusion with std=0.8 to model 'resnet18_confused_base'...
  Found backbone layer: res_net_backbone_confused_base
  Confused 20 Conv2D layers.
  Confusing final Dense layer: dense_confused_base
  Final Dens

## 5. SNISA + UnconLa Training Loop
Iterate through each shard. For each shard, clone the `base_confused_model`, train it on the shard's retain data, apply confusion, train more, apply confusion, and train again. Save each trained shard model.

In [ ]:
shard_models = [] # List to store the final trained model for each shard
histories = []    # List to store training history for each shard

num_stages = len(EPOCHS_PER_STAGE)
print(f'Starting SNISA + UnconLa training for {NUM_SHARDS} shards...')
print(f'Training cycle per shard: {" epochs -> confuse -> ".join(map(str, EPOCHS_PER_STAGE))} epochs')

for s in range(NUM_SHARDS):
    print(f'\n--- Training Shard {s+1}/{NUM_SHARDS} ---')
    shard_data = rt_datasets[s] # Batched retain data for this shard
    # Use the global validation set for consistent evaluation across shards
    shard_val_data = val_ds

    # Clone the base confused model for this shard's training run
    shard_model = clone_model(base_confused_model, strategy)
    # Assign a unique name to this shard's model and its layers
    shard_name = f'shard_{s}'
    change_model_names(shard_model, name=shard_name, suffix=f'_{shard_name}')
    print(f'Cloned base confused model for shard {s}. New name: {shard_model.name}')

    # Dictionary to accumulate history across training stages for this shard
    shard_history = {} # Initialize empty, keys will be added dynamically
    current_epoch = 0 # Track epochs for correct history plotting and continuation

    # Determine metric keys once before starting stages
    # Fit for one epoch to get history keys if not already known
    # (Or assume keys based on compilation if possible, but fitting one epoch is safer)
    print(f"Determining metric keys for shard {s}...")
    # Temporarily fit for 0 epochs on a small batch just to get the history object structure
    # This assumes shard_data is a tf.data.Dataset; take 1 batch
    temp_history = shard_model.fit(shard_data.take(1), validation_data=shard_val_data.take(1) if shard_val_data else None, epochs=1, verbose=0)
    available_keys = temp_history.history.keys()
    print(f"DEBUG: Available history keys: {available_keys}")

    acc_key = None
    val_acc_key = None
    loss_key = 'loss' # Usually standard
    val_loss_key = 'val_loss' # Usually standard

    # Find accuracy key
    if 'accuracy' in available_keys:
        acc_key = 'accuracy'
    elif 'sparse_categorical_accuracy' in available_keys:
        acc_key = 'sparse_categorical_accuracy'
    else:
        for key in available_keys:
            if 'accuracy' in key and 'val_' not in key:
                acc_key = key
                break
    if acc_key is None:
        # Fallback if no accuracy key found (should not happen with standard metrics)
        print(f"Warning: Could not find a standard accuracy key in history keys: {available_keys}. Training might proceed without accuracy tracking.")
        # Try to find *any* key if necessary, or raise error
        # raise KeyError(f"Could not find a suitable accuracy key in history keys: {available_keys}")


    # Find validation accuracy key (only if validation data is present)
    if shard_val_data:
        if 'val_accuracy' in available_keys:
            val_acc_key = 'val_accuracy'
        elif 'val_sparse_categorical_accuracy' in available_keys:
            val_acc_key = 'val_sparse_categorical_accuracy'
        else:
            for key in available_keys:
                if 'accuracy' in key and 'val_' in key:
                    val_acc_key = key
                    break
        if val_acc_key is None:
             print(f"Warning: Could not find a standard validation accuracy key in history keys: {available_keys}. Training might proceed without validation accuracy tracking.")
             # Fallback or error
             # raise KeyError(f"Could not find a suitable validation accuracy key in history keys: {available_keys}")
    else:
        val_acc_key = None # No validation data, no validation keys
        val_loss_key = None

    print(f"DEBUG: Determined keys - loss: {loss_key}, acc: {acc_key}, val_loss: {val_loss_key}, val_acc: {val_acc_key}")

    # Initialize shard_history with determined keys
    shard_history[loss_key] = []
    if acc_key: shard_history[acc_key] = []
    if val_loss_key: shard_history[val_loss_key] = []
    if val_acc_key: shard_history[val_acc_key] = []


    # --- Training Stages Loop ---
    for stage_idx, stage_epochs in enumerate(EPOCHS_PER_STAGE):
        print(f'\nShard {s} - Stage {stage_idx + 1}/{num_stages} Training ({stage_epochs} epochs)...')

        # Calculate target epoch for this stage
        target_epoch = current_epoch + stage_epochs

        history_stage = shard_model.fit(
            shard_data,
            validation_data=shard_val_data,
            epochs=target_epoch,
            initial_epoch=current_epoch,
            verbose=1
        )

        # Append history from this stage
        shard_history[loss_key].extend(history_stage.history[loss_key])
        if acc_key and acc_key in history_stage.history:
             shard_history[acc_key].extend(history_stage.history[acc_key])
        if val_loss_key and val_loss_key in history_stage.history:
             shard_history[val_loss_key].extend(history_stage.history[val_loss_key])
        if val_acc_key and val_acc_key in history_stage.history:
             shard_history[val_acc_key].extend(history_stage.history[val_acc_key])

        current_epoch = target_epoch # Update current epoch count

        # Apply confusion, except after the last stage
        if stage_idx < num_stages - 1:
            print(f'\nShard {s} - Applying confusion after stage {stage_idx + 1} (std={CONFUSION_STD})...')
            vision_confuser(shard_model, std=CONFUSION_STD)
        else:
            print(f'\nShard {s} - Finished final training stage.')


    shard_model_path = os.path.join(MODELS_DIR, f'shard_{s}_model.keras')
    print(f'\nSaving shard {s} model to {shard_model_path}...')
    shard_model.save(shard_model_path)

    shard_models.append(shard_model)
    histories.append(shard_history)
    print(f'--- Finished Training Shard {s+1}/{NUM_SHARDS} ---')

print(f'\nFinished training and saving all {NUM_SHARDS} shard models.')


Starting SNISA + UnconLa training for 4 shards...
Training cycle per shard: 1 epochs -> confuse -> 1 epochs -> confuse -> 1 epochs -> confuse -> 1 epochs -> confuse -> 2 epochs -> confuse -> 4 epochs

--- Training Shard 1/4 ---
Cloned and re-compiled model 'resnet18_confused_base' with explicit accuracy metric.
Updated layer names for model 'res_net_backbone_confused_base_shard_0' with suffix '_shard_0'.
Updated layer names for model 'shard_0' with suffix '_shard_0'.
Cloned base confused model for shard 0. New name: shard_0
Determining metric keys for shard 0...
Cloned and re-compiled model 'resnet18_confused_base' with explicit accuracy metric.
Updated layer names for model 'res_net_backbone_confused_base_shard_0' with suffix '_shard_0'.
Updated layer names for model 'shard_0' with suffix '_shard_0'.
Cloned base confused model for shard 0. New name: shard_0
Determining metric keys for shard 0...


W0000 00:00:1746106735.531220   63656 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


DEBUG: Available history keys: dict_keys(['loss', 'accuracy', 'val_loss', 'val_accuracy'])
DEBUG: Determined keys - loss: loss, acc: accuracy, val_loss: val_loss, val_acc: val_accuracy

Shard 0 - Stage 1/6 Training (1 epochs)...
88/88 [==============================] - 17s 190ms/step - loss: 17.0804 - accuracy: 0.1900 - val_loss: 2089.4487 - val_accuracy: 0.1215

Shard 0 - Applying confusion after stage 1 (std=0.08)...
Applying vision confusion with std=0.08 to model 'shard_0'...
  Found backbone layer: res_net_backbone_confused_base_shard_0

Shard 0 - Applying confusion after stage 1 (std=0.08)...
Applying vision confusion with std=0.08 to model 'shard_0'...
  Found backbone layer: res_net_backbone_confused_base_shard_0
  Confused 20 Conv2D layers.
  Confusing final Dense layer: dense_confused_base_shard_0
  Final Dense layer confused.

Shard 0 - Stage 2/6 Training (1 epochs)...
Epoch 2/2
  Confused 20 Conv2D layers.
  Confusing final Dense layer: dense_confused_base_shard_0
  Final D

W0000 00:00:1746106806.244570   63658 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


DEBUG: Available history keys: dict_keys(['loss', 'accuracy', 'val_loss', 'val_accuracy'])
DEBUG: Determined keys - loss: loss, acc: accuracy, val_loss: val_loss, val_acc: val_accuracy

Shard 1 - Stage 1/6 Training (1 epochs)...
88/88 [==============================] - 15s 169ms/step - loss: 17.5291 - accuracy: 0.1819 - val_loss: 1929.5328 - val_accuracy: 0.1213

Shard 1 - Applying confusion after stage 1 (std=0.08)...
Applying vision confusion with std=0.08 to model 'shard_1'...
  Found backbone layer: res_net_backbone_confused_base_shard_1

Shard 1 - Applying confusion after stage 1 (std=0.08)...
Applying vision confusion with std=0.08 to model 'shard_1'...
  Found backbone layer: res_net_backbone_confused_base_shard_1
  Confused 20 Conv2D layers.
  Confusing final Dense layer: dense_confused_base_shard_1
  Final Dense layer confused.

Shard 1 - Stage 2/6 Training (1 epochs)...
Epoch 2/2
  Confused 20 Conv2D layers.
  Confusing final Dense layer: dense_confused_base_shard_1
  Final D

W0000 00:00:1746106875.069305   63657 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


DEBUG: Available history keys: dict_keys(['loss', 'accuracy', 'val_loss', 'val_accuracy'])
DEBUG: Determined keys - loss: loss, acc: accuracy, val_loss: val_loss, val_acc: val_accuracy

Shard 2 - Stage 1/6 Training (1 epochs)...
88/88 [==============================] - 15s 167ms/step - loss: 17.3976 - accuracy: 0.1801 - val_loss: 2481.5388 - val_accuracy: 0.1123

Shard 2 - Applying confusion after stage 1 (std=0.08)...
Applying vision confusion with std=0.08 to model 'shard_2'...
  Found backbone layer: res_net_backbone_confused_base_shard_2

Shard 2 - Applying confusion after stage 1 (std=0.08)...
Applying vision confusion with std=0.08 to model 'shard_2'...
  Found backbone layer: res_net_backbone_confused_base_shard_2
  Confused 20 Conv2D layers.
  Confusing final Dense layer: dense_confused_base_shard_2
  Final Dense layer confused.

Shard 2 - Stage 2/6 Training (1 epochs)...
Epoch 2/2
  Confused 20 Conv2D layers.
  Confusing final Dense layer: dense_confused_base_shard_2
  Final D

W0000 00:00:1746106944.847170   63657 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


DEBUG: Available history keys: dict_keys(['loss', 'accuracy', 'val_loss', 'val_accuracy'])
DEBUG: Determined keys - loss: loss, acc: accuracy, val_loss: val_loss, val_acc: val_accuracy

Shard 3 - Stage 1/6 Training (1 epochs)...
88/88 [==============================] - 96s 1s/step - loss: 17.4187 - accuracy: 0.1774 - val_loss: 2135.6135 - val_accuracy: 0.1138

Shard 3 - Applying confusion after stage 1 (std=0.08)...
Applying vision confusion with std=0.08 to model 'shard_3'...
  Found backbone layer: res_net_backbone_confused_base_shard_3

Shard 3 - Applying confusion after stage 1 (std=0.08)...
Applying vision confusion with std=0.08 to model 'shard_3'...
  Found backbone layer: res_net_backbone_confused_base_shard_3
  Confused 20 Conv2D layers.
  Confusing final Dense layer: dense_confused_base_shard_3
  Confused 20 Conv2D layers.
  Confusing final Dense layer: dense_confused_base_shard_3
  Final Dense layer confused.

Shard 3 - Stage 2/6 Training (1 epochs)...
  Final Dense layer co

## 6. Build Ensemble Model
Combine the individually trained shard models into a single ensemble model. The ensemble's prediction is the average of the predictions from all shard models.

In [ ]:
def build_ensemble_model(shard_models, strategy, input_shape=(32, 32, 3), num_classes=10):
    "Builds an ensemble model that averages the outputs of shard models."
    if not shard_models:
        raise ValueError('Cannot build ensemble with no shard models.')
    print(f'Building ensemble model from {len(shard_models)} shard models...')
    
    with strategy.scope():
# Define the input layer, matching the input shape of the shard models
        input_tensor = keras.Input(shape=input_shape)

        # Get the output tensor from each shard model for the same input
        outputs = [model(input_tensor) for model in shard_models]

        # Use the Average layer to average the outputs (element-wise)
        average_output = layers.Average()(outputs)

        # Create the ensemble model
        ensemble_model = keras.Model(inputs=input_tensor, outputs=average_output, name='snisa_unconla_ensemble')

        # Compile the ensemble model for evaluation purposes
        ensemble_model.compile(
            optimizer='adam', # Optimizer choice doesn't impact evaluation
            loss='sparse_categorical_crossentropy', # Use the same loss as shard models
            metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
            jit_compile=True # Enable JIT if desired
        )
    print('Ensemble model built and compiled.')
    ensemble_model.summary()
    return ensemble_model

# Build the ensemble from the trained shard models
ensemble_model = build_ensemble_model(shard_models, strategy)


## 7. Evaluate Ensemble Model Accuracy
Measure the classification accuracy of the final ensemble model on the retain set (data it was trained on, excluding forget), the forget set (data we want to unlearn), and the test set (unseen data).

In [ ]:
# Define the datasets for evaluation
# Note: We use the full retain_ds and forget_ds here, not the shards
datasets_to_evaluate = [retain_ds, forget_ds, test_ds]
dataset_names = ['Retain', 'Forget', 'Test']

print("Evaluating the final ensemble model...")
# The evaluate_model function handles batching if datasets are unbatched
accuracy_results = evaluate_model(ensemble_model, datasets_to_evaluate, dataset_names)

print('\n--- Ensemble Accuracy Results ---')
for name, acc in accuracy_results.items():
    print(f'  {name} Set Accuracy: {acc * 100.0:.2f}%')
print('---------------------------------')


## 8. Perform Membership Inference Attack (MIA)
Train a simple logistic regression model to distinguish between samples from the forget set and the test set based *only* on the loss produced by the ensemble model for those samples. A lower MIA accuracy (closer to 50%) indicates that the model treats forget and test samples similarly, suggesting successful unlearning.

In [ ]:
# Perform MIA comparing the forget set vs the test set
# This uses the unbatched, normalized datasets (forget_set_normalized, test_ds_normalized)
# The function computes losses internally and trains the attack model.
print("\nPerforming Membership Inference Attack (Forget vs. Test)...")
mia_scores = perform_mia_on_forget_test(ensemble_model, forget_set_normalized, test_ds_normalized, batch_size=BATCH_SIZE)

if mia_scores is not None and not np.isnan(mia_scores).any():
    mean_mia_accuracy = np.mean(mia_scores)
    std_mia_accuracy = np.std(mia_scores)
    print(f'\n--- MIA Results ---')
    print(f'  Mean Accuracy: {mean_mia_accuracy * 100.0:.2f}%')
    print(f'  Standard Deviation: {std_mia_accuracy * 100.0:.2f}%')
    print('  (Closer to 50% is better for unlearning)')
    print('-------------------')
else:
    print('\nMIA could not be performed or resulted in NaN scores.')
    mean_mia_accuracy = np.nan # Assign NaN if MIA failed


## 9. Plot Results
Visualize the distribution of losses for the forget set and the test set. If unlearning is effective, these distributions should overlap significantly. Also, plot the training history (loss and accuracy vs. epoch) for each shard.

In [ ]:
# --- Plot Loss Distribution ---
# Compute losses for plotting distributions (can be time-consuming)
print('\nComputing losses for plotting loss distribution...')
try:
# Use the batched datasets (forget_ds, test_ds) for efficiency
    forget_losses_final = compute_losses(ensemble_model, forget_ds) 
    test_losses_final = compute_losses(ensemble_model, test_ds)     
    print('Loss computation complete.')

# Plot the loss distributions
    plot_loss_distribution(test_losses_final, forget_losses_final, 
                         label1='Test Set', label2='Forget Set', 
                         title=f'Ensemble Model Loss Distribution (MIA Acc: {mean_mia_accuracy:.3f})')
except Exception as e:
    print(f'Could not compute or plot loss distributions: {e}')

# --- Plot Training History Per Shard ---
print("\nPlotting training history for each shard...")
for i, history_data in enumerate(histories):
    acc_key_plot = 'accuracy' if 'accuracy' in history_data else 'sparse_categorical_accuracy'
    val_acc_key_plot = 'val_accuracy' if 'val_accuracy' in history_data else 'val_sparse_categorical_accuracy'
    loss_key_plot = 'loss'
    val_loss_key_plot = 'val_loss'
    
    history_for_plot = {
        'loss': history_data.get(loss_key_plot, []),
        'val_loss': history_data.get(val_loss_key_plot, []),
        'accuracy': history_data.get(acc_key_plot, []),
        'val_accuracy': history_data.get(val_acc_key_plot, [])
    }
    
    history_for_plot = {k: v for k, v in history_for_plot.items() if v}
    if not history_for_plot:
        print(f"Skipping plot for shard {i}: No history data found.")
        continue
    plot_training_history(history_for_plot, title=f'Shard {i} Training History (SNISA + UnconLa)')


## 10. Save Results
Save the key configuration parameters and evaluation metrics (accuracies, MIA score) to a file for later analysis or comparison.

In [ ]:
# Consolidate results into a dictionary
results_data = {
    'num_shards': NUM_SHARDS,
    'forget_fraction': FORGET_FRACTION,
    'confusion_std': CONFUSION_STD,
    'epochs_per_stage': EPOCHS_PER_STAGE,
    'total_epochs_per_shard': TOTAL_EPOCHS_PER_SHARD,
    'seed': SEED,
    'accuracy_retain': accuracy_results.get('Retain', np.nan),
    'accuracy_forget': accuracy_results.get('Forget', np.nan),
    'accuracy_test': accuracy_results.get('Test', np.nan),
    'mia_accuracy_forget_vs_test_mean': mean_mia_accuracy,
    'mia_accuracy_forget_vs_test_std': std_mia_accuracy if 'std_mia_accuracy' in locals() else np.nan
}

# Define filename using key parameters
results_filename = os.path.join(RESULTS_DIR, f'snisa_unconla_results_S{NUM_SHARDS}_std{CONFUSION_STD}.pkl')
print(f'\nSaving results to {results_filename}...')
try:
    with open(results_filename, 'wb') as f:
        pickle.dump(results_data, f)
    print('Results saved successfully.')
    print("\nResults Data:")
    for key, value in results_data.items():
        print(f"  {key}: {value}")
except Exception as e:
    print(f'Error saving results: {e}')
